# START HERE — reproduce the main results

One notebook, top to bottom. Each section is independent after section 1.
Runtime on CPU: sections 1–3 a few minutes, section 4 ~10 min, section 5 slow.

**Read `bal-acc` and `precision` together, never accuracy alone.** The majority
baseline is 0.833 (2015 PD-vs-ET) and 0.908 (PADS), so raw accuracy is
misleading, and `class_weight="balanced"` trades precision for recall — a
respectable balanced accuracy can hide a precision collapse.

| axis | best result | where |
|---|---|---|
| **N vs Tremor** | internal 0.814, **external 0.736 / AUC 0.783** on PADS | §4 |
| PD vs ET | bal-acc 0.730, **ET precision 0.393**, F1 0.500 | §3 |

Full background: `reports/` — especially `final_results.md`,
`pads_label_bug.md`, `quaternion_session_verdict.md`.

## 1. Setup — loads all three cohorts and verifies them

In [1]:
import sys, os
if os.path.basename(os.getcwd()) == "tfbench": os.chdir("..")
sys.path.insert(0, os.getcwd())
import numpy as np, matplotlib.pyplot as plt
from tremor.quaternion_data import load_quaternion_recordings
from pdetn.load_2025 import load_2025_all
from pdetn.crossdataset import load_pads_extracted

loc  = load_quaternion_recordings("Data", action="REST", mode="angular_velocity")
new  = load_2025_all(conditions=("REST",))                 # segmented by default
try:
    pads = load_pads_extracted("pads_relaxed", task="Relaxed")
except FileNotFoundError:
    pads = None; print("pads_relaxed/ not present -- see docs/EXTRACT_PADS_RELAXED.md")

for nm, r in (("2015 REST", loc), ("NewData REST", new), ("PADS Relaxed", pads)):
    if r is None: continue
    y = np.array([x.y for x in r])
    print(f"{nm:>14}: {len(r):>4} recordings | patients N/PD/ET = "
          f"{[len({x.subject for x in r if x.y==k}) for k in (0,1,2)]}")
print("\nPADS must read [79, 276, 28] -- if ET reads 41 the label filter has regressed.")

     2015 REST:  275 recordings | patients N/PD/ET = [61, 75, 16]
  NewData REST:  102 recordings | patients N/PD/ET = [27, 25, 6]
  PADS Relaxed:  766 recordings | patients N/PD/ET = [79, 276, 28]

PADS must read [79, 276, 28] -- if ET reads 41 the label filter has regressed.


## 2. Mean and max frequency across every cohort

Three tables: per-class medians, PD-vs-ET direction and significance, and
classification from those two features alone.

In [2]:
from tfbench.frequency_report import report
S = report("Data")

### A. Median MAX and MEAN frequency (Hz) per class
   cohort    condition                 N                PD                ET
                              max / mean        max / mean        max / mean


     2015         REST       7.03 / 8.13       5.47 / 7.09       6.15 / 7.45


     2015          OUT       7.81 / 8.29       6.64 / 7.35       6.45 / 7.25


     2015         WING       3.71 / 7.15       4.49 / 6.51       5.66 / 6.63


  NewData   REST (seg)                 —                 —       5.86 / 6.59


...

     2015         WING     PD-vs-ET    76  0.829    0.462  0.499  0.147  0.385   0.213 [0.05,0.37]


     PADS      Relaxed  N-vs-Tremor   383  0.794    0.468  0.499  0.774  0.507   0.612 [0.56,0.66]


     PADS      Relaxed     PD-vs-ET   304  0.908    0.669  0.759  0.176  0.643   0.277 [0.18,0.38]


     PADS  StretchHold  N-vs-Tremor   383  0.794    0.612  0.664  0.855  0.641   0.733 [0.69,0.77]


     PADS  StretchHold     PD-vs-ET   304  0.908    0.703  0.775  0.202  0.679   0.311 [0.20,0.41]


## 3. The best PD-vs-ET model

2015 REST, lower_arm, stft512, all 10 descriptors, threshold 0.5.
Reported per class, because ET precision is the number that matters.

In [3]:
from collections import defaultdict
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneGroupOut, cross_val_predict
from tfbench.transforms import METHODS
from tfbench.descriptors import describe, DESCRIPTOR_NAMES
from tfbench.merged import per_class_report, bal_acc

def descriptor_table(recs, method="stft512", ch=slice(3, 6)):
    fn = METHODS[method]; rows, lab = defaultdict(list), {}
    for r in recs:
        x = r.x[ch] if r.x.shape[0] > 3 else r.x
        rows[r.subject].append([describe(*fn(x))[c] for c in DESCRIPTOR_NAMES])
        lab[r.subject] = r.y
    p = sorted(rows)
    return (np.nan_to_num(np.array([np.mean(rows[k], 0) for k in p])),
            np.array([lab[k] for k in p]), np.array(p))

def clf():
    return make_pipeline(StandardScaler(),
                         LogisticRegression(max_iter=5000, class_weight="balanced"))

X, y, g = descriptor_table(loc)
pred3 = cross_val_predict(clf(), X, y, groups=g, cv=LeaveOneGroupOut())
per_class_report(y, pred3, tag="=== 2015 REST, 3-class, patient-level LOSO ===")

from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
k = y != 0
pr = cross_val_predict(clf(), X[k], (y[k]==2).astype(int), groups=g[k],
                       cv=LeaveOneGroupOut(), method="predict_proba")[:, 1]
yy, pd_ = (y[k]==2).astype(int), (pr >= 0.5).astype(int)
print(f"\nPD-vs-ET binary: bal-acc {bal_acc(yy,pd_):.3f} | AUC {roc_auc_score(yy,pr):.3f} "
      f"| precision {precision_score(yy,pd_):.3f} | recall {recall_score(yy,pd_):.3f} "
      f"| F1 {f1_score(yy,pd_):.3f}   (majority baseline {max(yy.mean(),1-yy.mean()):.3f})")


=== 2015 REST, 3-class, patient-level LOSO ===
   class  precision   recall      F1  support  prevalence
       N      0.672    0.705   0.688       61       0.401
      PD      0.750    0.560   0.641       75       0.493
      ET      0.219    0.438   0.292       16       0.105
   macro      0.547    0.567   0.540      152
  confusion (rows=true, cols=pred): [[43, 10, 8], [16, 42, 17], [5, 4, 7]]



PD-vs-ET binary: bal-acc 0.730 | AUC 0.729 | precision 0.393 | recall 0.688 | F1 0.500   (majority baseline 0.824)


## 4. The externally validated result — N vs Tremor

Train on 2015 + NewData, score once on PADS. This is the axis that transfers.

In [4]:
if pads is not None:
    from tfbench.merged import merge, loso, report as mreport, external_validate
    from tremor.quaternion_data import load_quaternion_recordings as _lq
    from pdetn.load_2025 import load_2025

    locO = _lq("Data", action="OUT", mode="angular_velocity")
    newO = load_2025(mode="angular_velocity", conditions=("OUT",))
    try:
        ext = load_pads_extracted("pads_stretchhold", task="StretchHold")
    except FileNotFoundError:
        ext = None
    (Xm, ym, gm), info = merge(locO, newO, "stft512")
    print(f"NewData device probe: AUC {info['identity_auc']:.3f} (|dev| {info['deviation']:.3f})")
    ya = (ym != 0).astype(int)
    prm, predm = loso(Xm, ya, gm)
    mreport(ya, prm, predm, gm, "N-vs-Tremor merged LOSO (internal)")
    if ext is not None:
        Xp, yp, gp = descriptor_table(ext, "stft512", slice(0, 3))
        external_validate(Xm, ya, Xp, (yp != 0).astype(int), gp,
                          "N-vs-Tremor -> PADS (external)")

NewData device probe: AUC 0.200 (|dev| 0.300)


            N-vs-Tremor merged LOSO (internal) n=157 pos= 96 | bal-acc 0.814 | AUC 0.903 | P 0.884 R 0.792 | F1 0.835 [0.77,0.89] | maj 0.611


                N-vs-Tremor -> PADS (external) n=383 pos=304 | bal-acc 0.736 | AUC 0.783 | P 0.928 R 0.674 | F1 0.781 [0.74,0.82] | maj 0.794


## 5. Optional — method benchmark and deep models

Slow. The 12-method benchmark caches its tables, so the second run is fast.

In [5]:
# 12 time-frequency methods, BH-corrected screen + paired-CI ranking
# from tfbench import benchmark as B
# from tfbench.cache import load_or_build
# tables = load_or_build(loc, path="artifacts/tfbench_tables.npz")
# B.screen(tables, axis="PD_vs_ET", top=12)
# B.rank_methods(tables, axis="PD_vs_ET", reference="welch", n_boot=20000)

# Deep models. NOTE: on PD-vs-ET these sit at chance (AUC 0.475-0.517 across
# six configurations) -- 16 ET is too few for ~1e5 parameters. See
# reports/deep_pdvset_2015.md before spending time here.
# from tfbench.deep import compare
# compare(loc, methods=["stft512"], archs=["tremor_bilstm"], axis="N_vs_Tremor",
#         seeds=(0,), n_splits=3, epochs=20)

## 6. Best PD-vs-ET model — small net on the kinetic task

The strongest result in the project, and the most fragile. Two things drive it:

* **Task.** `DRINK` (and `FINGER_NOSE`) are kinetic-tremor manoeuvres. ET is a
  kinetic tremor; rest and postural tasks sit at or below chance for PD-vs-ET.
* **Architecture.** A BiLSTM over the **frequency** axis of a time-averaged
  spectrum — 9,090 parameters. The same family over the **time** axis of a
  spectrogram sits at chance, as do ResNet18 / WideResNet / ViT at 11–86 M
  parameters. Capacity is not the variable; what the model reads is.

**6 ET subjects.** Treat as a lead, not a result.

In [ ]:
from tfbench.small_nets import evaluate
from pdetn.load_2025 import load_2025_all
import torch; torch.set_num_threads(1)     # tiny tensors: threads hurt

drink = load_2025_all(conditions=("DRINK",))
print(f"{'class_weight':>14}{'n':>5}{'ET':>4}{'bal-acc':>9}{'AUC':>7}{'prec':>7}{'rec':>7}{'F1':>7}")
for cw in (False, True):
    r = evaluate(drink, axis="PD_vs_ET", class_weight=cw)
    print(f"{str(cw):>14}{r['n']:>5}{r['n_pos']:>4}{r['bal_acc']:>9.3f}{r['auc']:>7.3f}"
          f"{r['precision']:>7.3f}{r['recall']:>7.3f}{r['f1']:>7.3f}")

Class weighting is a **trade, not an improvement** — it buys recall and costs
precision on this architecture. Report both; pick by application (screening
wants recall, differential diagnosis wants precision).

### What has been ruled out

| attempt | result |
|---|---|
| pool 2015 + NewData + PADS (16 → 50 ET) | precision 0.393 → 0.163 |
| BiLSTM over time on spectrogram | AUC ~0.52 across 6 configs |
| ResNet18 / WideResNet / ViT | AUC 0.47–0.54 |
| per-axis x/y/z fusion | worse than averaging the axes |
| TCN(freq)+BiLSTM(time) hybrid | bal-acc 0.52–0.62 |
| temporal descriptors, multi-sensor | significantly worse |

Details in `reports/`. The binding constraint is 6 ET on this task —
**PADS `DrinkGlas` has 28** and is the experiment that decides whether any of
this is real.

## What this reproduces, and what it does not

**Works:** N-vs-Tremor, internally and on an unseen cohort.

**Does not work, and is documented as such:** ET identification. Best ET
precision anywhere is 0.393 (binary) / 0.219 (3-class) on 16 ET subjects.
Measured dead ends, each with a report in `reports/`:

| attempt | ET precision |
|---|---|
| baseline — 2015 only, 10 descriptors | **0.393** |
| pool all three cohorts (16 → 50 ET) | 0.163 |
| BiLSTM, best of 6 configurations | 0.180 |
| max + mean frequency only | 0.220 |

The constraint is 16 ET subjects from one consistent population — not the
model, the features, or the total subject count.